# 📺 YouTube → Dailymotion Uploader (Improved)

This notebook will:

1. 🔧 Install tools and mount your Google Drive
2. ⬇️ Download a YouTube video straight to Google Drive — paste the link, give it a name, pick a quality, and run
3. 🔑 Log in to Dailymotion with a simple credentials form
4. ✂️ Split any video already in your Drive by duration (not size) — point it at the file, it splits into parts under Dailymotion's 2-hour limit
5. 📤 Upload any folder of parts already in your Drive to Dailymotion, with live progress, ETA, and a graceful stop if you hit Dailymotion's daily upload limit

Steps 4 and 5 are fully independent — you can run them on their own, any day, pointing at whatever's already sitting in your Drive, even if you didn't download or split it in this same session.

In [ ]:
#@title 🔧 Step 1: Setup — Install Tools & Mount Google Drive
#@markdown Run this first. It installs the required tools and connects your Google Drive.

import sys, subprocess

print("📦 Installing yt-dlp, requests, tqdm, requests-toolbelt, ipywidgets, gdown...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yt-dlp", "requests", "tqdm", "requests-toolbelt", "ipywidgets", "gdown"], check=True)

from google.colab import drive
print("\n📂 Mounting Google Drive...")
drive.mount('/content/drive')

print("\n✅ Setup complete! Drive is mounted at /content/drive")

In [ ]:
#@title ⬇️ Step 2: Download YouTube Video
#@markdown Paste the link, give it a name, pick a quality, and run this cell — it downloads straight to Drive.

YOUTUBE_URL = "" #@param {type:"string"}
CUSTOM_TITLE = "" #@param {type:"string"}
#@markdown Leave Custom Title blank to use the video's original YouTube title.
VIDEO_QUALITY = "1080p" #@param ["Best Available", "1080p", "720p", "480p"]
DRIVE_OUTPUT_FOLDER = "/content/drive/MyDrive/YouTube_Dailymotion_Uploads" #@param {type:"string"}

import os
from yt_dlp import YoutubeDL
from tqdm.notebook import tqdm

if not YOUTUBE_URL:
    raise ValueError("⚠️ Please paste a YouTube URL above.")

os.makedirs(DRIVE_OUTPUT_FOLDER, exist_ok=True)

QUALITY_FORMATS = {
    "Best Available": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best",
    "1080p": "bestvideo[height<=1080][ext=mp4]+bestaudio[ext=m4a]/best[height<=1080][ext=mp4]/best[height<=1080]",
    "720p": "bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]/best[height<=720][ext=mp4]/best[height<=720]",
    "480p": "bestvideo[height<=480][ext=mp4]+bestaudio[ext=m4a]/best[height<=480][ext=mp4]/best[height<=480]",
}
FORMAT_STRING = QUALITY_FORMATS[VIDEO_QUALITY]

output_template = os.path.join(DRIVE_OUTPUT_FOLDER, "%(title)s.%(ext)s")

_state = {"bar": None}

def _progress_hook(d):
    if d["status"] == "downloading":
        total = d.get("total_bytes") or d.get("total_bytes_estimate")
        downloaded = d.get("downloaded_bytes", 0)
        if _state["bar"] is None and total:
            _state["bar"] = tqdm(total=total, unit="B", unit_scale=True, unit_divisor=1024, desc="⬇️ Downloading")
        if _state["bar"] is not None:
            _state["bar"].n = downloaded
            speed = d.get("speed")
            eta = d.get("eta")
            postfix = {}
            if speed:
                postfix["speed"] = f"{speed/1024/1024:.2f} MB/s"
            if eta is not None:
                mins, secs = divmod(int(eta), 60)
                postfix["ETA"] = f"{mins}m {secs}s"
            _state["bar"].set_postfix(postfix)
            _state["bar"].refresh()
    elif d["status"] == "finished":
        if _state["bar"] is not None:
            _state["bar"].n = _state["bar"].total
            _state["bar"].refresh()
            _state["bar"].close()
            _state["bar"] = None
        print("🔄 Merging audio/video (if needed)...")

ydl_opts = {
    "format": FORMAT_STRING,
    "merge_output_format": "mp4",
    "outtmpl": output_template,
    "progress_hooks": [_progress_hook],
    "quiet": True,
    "no_warnings": True,
}

print(f"⬇️  Downloading in {VIDEO_QUALITY}...\n")
with YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(YOUTUBE_URL, download=True)
    VIDEO_PATH = ydl.prepare_filename(info)
    base, _ext = os.path.splitext(VIDEO_PATH)
    if os.path.exists(base + ".mp4"):
        VIDEO_PATH = base + ".mp4"

VIDEO_TITLE = CUSTOM_TITLE.strip() if CUSTOM_TITLE.strip() else os.path.splitext(os.path.basename(VIDEO_PATH))[0]

if CUSTOM_TITLE.strip():
    new_path = os.path.join(DRIVE_OUTPUT_FOLDER, VIDEO_TITLE + ".mp4")
    if os.path.abspath(new_path) != os.path.abspath(VIDEO_PATH):
        os.rename(VIDEO_PATH, new_path)
        VIDEO_PATH = new_path

size_mb = os.path.getsize(VIDEO_PATH) / (1024 * 1024)
print(f"\n✅ Downloaded: {os.path.basename(VIDEO_PATH)}")
print(f"   • Size: {size_mb:.1f} MB")
print(f"   • Saved to: {VIDEO_PATH}")

In [ ]:
#@title 🔑 Step 3: Login & Authenticate with Dailymotion
#@markdown Fill in your Dailymotion API credentials below, then click **Login & Authenticate**.
#@markdown Nothing you type here gets written into this notebook file — the fields only hold values in memory for this session.

import ipywidgets as widgets
from IPython.display import display
import requests

_box_style = {'description_width': '120px'}
_box_layout = widgets.Layout(width='420px')

_client_id_box = widgets.Text(description="Client ID:", placeholder="Your API key", style=_box_style, layout=_box_layout)
_client_secret_box = widgets.Password(description="Client Secret:", placeholder="Your API secret", style=_box_style, layout=_box_layout)
_username_box = widgets.Text(description="Email:", placeholder="Your Dailymotion email", style=_box_style, layout=_box_layout)
_password_box = widgets.Password(description="Password:", placeholder="Your Dailymotion password", style=_box_style, layout=_box_layout)
_login_button = widgets.Button(description="🔐 Login & Authenticate", button_style="success")
_status_output = widgets.Output()

ACCESS_TOKEN = None

def _do_login(_):
    global ACCESS_TOKEN
    with _status_output:
        _status_output.clear_output()
        print("🔐 Authenticating with Dailymotion...")
        token_url = "https://api.dailymotion.com/oauth/token"
        data = {
            "grant_type": "password",
            "client_id": _client_id_box.value,
            "client_secret": _client_secret_box.value,
            "username": _username_box.value,
            "password": _password_box.value,
            "scope": "manage_videos userinfo",
        }
        try:
            response = requests.post(token_url, data=data)
            if response.status_code == 200:
                ACCESS_TOKEN = response.json()["access_token"]
                print("✅ Logged in and authenticated successfully!")
            else:
                print(f"❌ Authentication failed: {response.status_code} - {response.text}")
        except Exception as e:
            print(f"❌ Error: {e}")

_login_button.on_click(_do_login)

display(widgets.VBox([_client_id_box, _client_secret_box, _username_box, _password_box, _login_button, _status_output]))

In [ ]:
#@title ✂️ Step 4: Split a Video by Duration
#@markdown Point this at any video already in your Drive — a folder path or a Google Drive share link. It doesn't have to be one you downloaded in Step 2.

VIDEO_DRIVE_PATH = "" #@param {type:"string"}
#@markdown Example path: `/content/drive/MyDrive/Folder/video.mp4` — or paste a Drive share link instead.
START_PART_NUMBER = 1 #@param {type:"integer"}
#@markdown Part numbering starts from this — useful if you're continuing a previous split.
SPLIT_HOURS = 1 #@param {type:"integer"}
SPLIT_MINUTES = 55 #@param {type:"integer"}
#@markdown Dailymotion's max length is 2 hours, so keep this under that.

import os, re, json, subprocess, shutil
from tqdm.notebook import tqdm

def resolve_drive_input(value, workdir):
    value = value.strip()
    if not value:
        raise ValueError("⚠️ Please provide a Drive path or a Drive share link above.")
    if value.startswith("http") and "drive.google.com" in value:
        match = re.search(r"/d/([a-zA-Z0-9_-]+)", value) or re.search(r"[?&]id=([a-zA-Z0-9_-]+)", value)
        if not match:
            raise ValueError("❌ Couldn't find a file ID in that Drive link.")
        file_id = match.group(1)
        print(f"🔗 Detected a Drive share link — downloading via file ID {file_id}...")
        import gdown
        os.makedirs(workdir, exist_ok=True)
        local_path = gdown.download(id=file_id, output=os.path.join(workdir, "downloaded_input.mp4"), quiet=False)
        if not local_path:
            raise Exception("❌ Failed to download the file from the Drive link.")
        return local_path
    if os.path.exists(value):
        return value
    raise FileNotFoundError(f"❌ Couldn't find a file at: {value}")

INPUT_VIDEO_PATH = resolve_drive_input(VIDEO_DRIVE_PATH, "/content/drive/MyDrive/YouTube_Dailymotion_Uploads/_downloaded_links")
VIDEO_TITLE = os.path.splitext(os.path.basename(INPUT_VIDEO_PATH))[0]
print(f"🎬 Using file: {INPUT_VIDEO_PATH}")
print(f"📝 Detected title: {VIDEO_TITLE}")

SPLIT_DURATION_SECONDS = SPLIT_HOURS * 3600 + SPLIT_MINUTES * 60

def get_duration(filepath):
    cmd = ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", filepath]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    return float(json.loads(result.stdout)["format"]["duration"])

def split_video_by_duration(input_path, output_dir, chunk_seconds, title, start_number):
    total_duration = get_duration(input_path)
    os.makedirs(output_dir, exist_ok=True)

    if total_duration <= chunk_seconds:
        print("ℹ️ Video is already within the length limit — no split needed.")
        single_path = os.path.join(output_dir, f"{title} - Part {start_number:03d}.mp4")
        if os.path.abspath(single_path) != os.path.abspath(input_path):
            shutil.copy(input_path, single_path)
        return [single_path]

    num_parts = -(-int(total_duration) // chunk_seconds)
    h, m = chunk_seconds // 3600, (chunk_seconds % 3600) // 60
    print(f"🎬 Video is {total_duration/3600:.2f}h long → splitting into {num_parts} part(s) of up to {h}h {m}m each, starting at Part {start_number}...\n")

    tmp_pattern = os.path.join(output_dir, "_raw_part_%03d.mp4")
    split_cmd = [
        "ffmpeg", "-y", "-i", input_path,
        "-c", "copy",
        "-map", "0",
        "-segment_time", str(chunk_seconds),
        "-f", "segment",
        "-reset_timestamps", "1",
        "-progress", "pipe:1", "-nostats",
        tmp_pattern,
    ]

    process = subprocess.Popen(split_cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, bufsize=1)

    pbar = tqdm(total=round(total_duration), unit="s", desc="✂️ Splitting")
    last_sec = 0
    for line in process.stdout:
        line = line.strip()
        if line.startswith("out_time_ms="):
            try:
                microseconds = int(line.split("=")[1])
                current_sec = microseconds / 1_000_000
                delta = current_sec - last_sec
                if delta > 0:
                    pbar.update(min(delta, max(0, pbar.total - pbar.n)))
                    last_sec = current_sec
                    remaining = max(0, total_duration - current_sec)
                    pbar.set_postfix({"ETA": f"{int(remaining)}s"})
            except ValueError:
                pass
        elif line == "progress=end":
            pbar.n = pbar.total
            pbar.refresh()
    process.wait()
    pbar.close()

    if process.returncode != 0:
        raise Exception("❌ ffmpeg failed while splitting the video.")

    raw_parts = sorted([f for f in os.listdir(output_dir) if f.startswith("_raw_part_") and f.endswith(".mp4")])
    final_parts = []
    for i, raw_name in enumerate(raw_parts):
        part_num = start_number + i
        final_path = os.path.join(output_dir, f"{title} - Part {part_num:03d}.mp4")
        os.rename(os.path.join(output_dir, raw_name), final_path)
        final_parts.append(final_path)

    print(f"\n✅ Split into {len(final_parts)} part(s) — the last part is simply whatever time remains:")
    for p in final_parts:
        print(f"   • {os.path.basename(p)}")
    return final_parts

parts_dir = os.path.join(os.path.dirname(INPUT_VIDEO_PATH), "parts")
video_parts = split_video_by_duration(INPUT_VIDEO_PATH, parts_dir, SPLIT_DURATION_SECONDS, VIDEO_TITLE, START_PART_NUMBER)
print(f"\n📦 Parts saved to: {parts_dir}")
print(f"📦 Ready to upload {len(video_parts)} part(s).")

In [ ]:
#@title 📤 Step 5: Upload Parts to Dailymotion
#@markdown Point this at the Drive folder (or single file) containing the part(s) to upload.

PARTS_DRIVE_PATH = "" #@param {type:"string"}
#@markdown Example: `/content/drive/MyDrive/YouTube_Dailymotion_Uploads/parts`
CUSTOM_TITLE_OVERRIDE = "" #@param {type:"string"}
#@markdown Leave blank to use each file's own name as the title (works automatically with names from Step 4).
DAILYMOTION_CHANNEL = "news" #@param ["news", "music", "sport", "tech", "creation", "auto"]
DAILYMOTION_TAGS = "youtube,upload" #@param {type:"string"}

import os, re, time, requests
from requests_toolbelt.multipart.encoder import MultipartEncoder, MultipartEncoderMonitor
from tqdm.notebook import tqdm

class DailyLimitReached(Exception):
    pass

if ACCESS_TOKEN is None:
    raise ValueError("⚠️ Please complete Step 3 (Login & Authenticate) first.")

if not PARTS_DRIVE_PATH.strip():
    raise ValueError("⚠️ Please provide a Drive folder or file path above.")

PARTS_DRIVE_PATH = PARTS_DRIVE_PATH.strip()

if os.path.isdir(PARTS_DRIVE_PATH):
    files = sorted([f for f in os.listdir(PARTS_DRIVE_PATH) if f.lower().endswith(".mp4")])
    part_paths = [os.path.join(PARTS_DRIVE_PATH, f) for f in files]
elif os.path.isfile(PARTS_DRIVE_PATH):
    part_paths = [PARTS_DRIVE_PATH]
else:
    raise FileNotFoundError(f"❌ Couldn't find: {PARTS_DRIVE_PATH}")

if not part_paths:
    raise Exception("❌ No .mp4 files found at that location.")

print(f"📦 Found {len(part_paths)} part(s) to upload:")
for p in part_paths:
    print(f"   • {os.path.basename(p)}")

PART_PATTERN = re.compile(r"^(.*) - Part (\d+)$")

def title_and_part_for(filepath):
    name = os.path.splitext(os.path.basename(filepath))[0]
    match = PART_PATTERN.match(name)
    if match:
        base_title, part_num = match.group(1), int(match.group(2))
    else:
        base_title, part_num = name, None
    if CUSTOM_TITLE_OVERRIDE.strip():
        base_title = CUSTOM_TITLE_OVERRIDE.strip()
    return base_title, part_num

DAILY_LIMIT_HINTS = ["limit", "quota", "too many", "daily", "rate limit"]

def is_daily_limit_error(response):
    text = (response.text or "").lower()
    return response.status_code in (403, 429) and any(hint in text for hint in DAILY_LIMIT_HINTS)

def upload_to_dailymotion(file_path, title, part_number, token, channel, tags):
    headers = {"Authorization": f"Bearer {token}"}

    res = requests.get("https://api.dailymotion.com/file/upload", headers=headers)
    if is_daily_limit_error(res):
        raise DailyLimitReached(res.text)
    res.raise_for_status()
    upload_url = res.json()["url"]

    file_size = os.path.getsize(file_path)
    pbar = tqdm(total=file_size, unit="B", unit_scale=True, unit_divisor=1024,
                desc=f"⬆️ Uploading {os.path.basename(file_path)}")
    start_time = time.time()

    def _callback(monitor):
        pbar.n = monitor.bytes_read
        elapsed = time.time() - start_time
        if elapsed > 0 and monitor.bytes_read > 0:
            speed = monitor.bytes_read / elapsed
            remaining_bytes = monitor.bytes_total - monitor.bytes_read
            eta_sec = remaining_bytes / speed if speed > 0 else 0
            mins, secs = divmod(int(eta_sec), 60)
            pbar.set_postfix({"speed": f"{speed/1024/1024:.2f} MB/s", "ETA": f"{mins}m {secs}s"})
        pbar.refresh()

    with open(file_path, "rb") as f:
        encoder = MultipartEncoder(fields={"file": (os.path.basename(file_path), f, "video/mp4")})
        monitor = MultipartEncoderMonitor(encoder, _callback)
        res = requests.post(upload_url, data=monitor, headers={"Content-Type": monitor.content_type})
        if is_daily_limit_error(res):
            pbar.close()
            raise DailyLimitReached(res.text)
        res.raise_for_status()
        video_url = res.json()["url"]

    pbar.n = file_size
    pbar.refresh()
    pbar.close()

    print("   📝 Publishing video...")
    final_title = f"{title} - Part {part_number}" if part_number else title

    publish_data = {
        "url": video_url,
        "title": final_title,
        "channel": channel,
        "tags": tags,
        "published": "true",
    }
    res = requests.post("https://api.dailymotion.com/me/videos", headers=headers, data=publish_data)
    if is_daily_limit_error(res):
        raise DailyLimitReached(res.text)
    res.raise_for_status()
    video_id = res.json()["id"]
    print(f"   ✅ Published as '{final_title}' → ID: {video_id}")
    return video_id, final_title

uploaded = []
remaining = list(part_paths)

for file_path in part_paths:
    title, part_num = title_and_part_for(file_path)
    print(f"\n--- {os.path.basename(file_path)} ---")
    try:
        vid_id, vid_title = upload_to_dailymotion(file_path, title, part_num, ACCESS_TOKEN, DAILYMOTION_CHANNEL, DAILYMOTION_TAGS)
        uploaded.append((vid_title, vid_id))
        remaining.remove(file_path)
    except DailyLimitReached:
        print("\n🚫 Dailymotion's daily upload limit has been reached.")
        print(f"   Uploaded {len(uploaded)} of {len(part_paths)} part(s) before hitting the limit.")
        print("   Remaining part(s) not uploaded:")
        for r in remaining:
            print(f"     • {os.path.basename(r)}")
        print("\n   👉 Wait ~24 hours, then re-run this Step 5 cell pointing at the same folder.")
        print("      Move or delete the already-uploaded files first so you don't upload duplicates.")
        break

if uploaded:
    print("\n🎉 Upload summary:\n")
    for title, vid_id in uploaded:
        print(f"   • {title}: https://www.dailymotion.com/video/{vid_id}")